In [ ]:
import pandas as pd
from matplotlib import pyplot as plt

from pathlib import Path
from time import strftime

from keras.layers import Flatten , Dense  , Embedding , Input , Concatenate , Dropout , BatchNormalization 
from keras.layers import  LeakyReLU
from keras.optimizers import Adam 
from keras.callbacks import EarlyStopping , TensorBoard , ReduceLROnPlateau ,ReduceLROnPlateau
from keras.losses import BinaryCrossentropy
from keras.models import Model
from tensorflow.random import set_seed

In [ ]:
x_train = pd.read_csv('./datas/Out_Stage3/x_train')
x_valid = pd.read_csv('./datas/Out_Stage3/x_valid')
y_train = pd.read_csv('./datas/Out_Stage3/y_train')
y_valid = pd.read_csv('./datas/Out_Stage3/y_valid')
numeric_cols = ['PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6','AGE','BILL_AMT1','BILL_AMT2',
                'BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6','PAY_AMT1','PAY_AMT2','PAY_AMT3','PAY_AMT4','PAY_AMT5','PAY_AMT6']

set_seed(42)

In [ ]:
input_marriage = Input(shape=(1,) , name='marriage')
embeding_marriage = Embedding(4,4 // 2 ,name = 'marriage_emb')(input_marriage)
layer_marriage = Flatten()(embeding_marriage)

input_education = Input(shape=(1,) , name='education')
embeding_education = Embedding(7,7 // 2 ,name = 'education_emb')(input_education)
layer_education = Flatten()(embeding_education)

input_sex = Input(shape=(1,) , name='sex')
embeding_sex = Embedding(2,2 // 2 ,name = 'sex_emb')(input_sex)
layer_sex = Flatten()(embeding_sex)

numeric_input = Input(shape=(19,))
layer_numeric = Dense(32, activation='relu')(numeric_input)


concat = Concatenate()([layer_marriage,layer_sex,layer_education ,layer_numeric])
wide_input = Concatenate()([layer_numeric])

hidden_wide_layer = Dense(1,use_bias=False)

output = Dense(1,activation='sigmoid')

tuned_params = {
          "n_neurons" : 64,
          "l2_rate" : 1.4792092042502516e-05,
          "activation" : "relu",
          "n_dropout" : 0.4,
          "n_hidden" : 5
     }

reg_params = {'l2_rate': 1.47e-05, 'dropout_rate': 0.4}   

wide_input = BatchNormalization()(wide_input)
layer_wide = Dense(1 , use_bias=False)(wide_input)

layer_deep = Dense(tuned_params['n_neurons'])(concat)
layer_deep = BatchNormalization()(layer_deep)
layer_deep = LeakyReLU()(layer_deep)
layer_deep = Dropout(reg_params['dropout_rate'])(layer_deep)

for _ in range(tuned_params['n_hidden']-1):
     layer_deep = Dense(tuned_params['n_neurons'])(layer_deep)
     layer_deep = BatchNormalization()(layer_deep)
     layer_deep =  LeakyReLU()(layer_deep)
     layer_deep = Dropout(reg_params['dropout_rate'])(layer_deep)


layer_main = Concatenate()([layer_wide,layer_deep])

layer_output = output(layer_main)

model_tuned = Model(inputs=[input_sex, input_marriage , input_education , numeric_input] , outputs =[layer_output])

model_tuned.compile(optimizer=Adam(learning_rate=1e-3 , weight_decay=reg_params['l2_rate']) , loss=BinaryCrossentropy() ,metrics=['accuracy'])




In [ ]:
def get_run_logdir(root_logdir="my_logs"):
    return Path(root_logdir) / strftime("run_%Y_%m_%d_%H_%M_%S")


earlyStopping = EarlyStopping('val_loss' ,patience=3)
tensorboard_cb = TensorBoard(get_run_logdir() , profile_batch=(100,200))
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

tune_regularization_model_h = model_tuned.fit(
     [x_train['SEX'],x_train['MARRIAGE'],x_train['EDUCATION'],x_train[numeric_cols]] ,
       y_train  ,
         epochs=50 ,
  validation_data=(
      [x_valid['SEX'],x_valid['MARRIAGE'],x_valid['EDUCATION'],x_valid[numeric_cols]],
      y_valid) ,
      callbacks=[earlyStopping , tensorboard_cb , lr_scheduler]
        )

In [ ]:
model_tuned.save('model_brain.h5')